# AI Agent Implementation

## Importing all the necessary Libraries

In [1]:
from dotenv import load_dotenv
import pandas as pd
import numpy as np
from langchain.vectorstores import FAISS
from langchain.llms import Ollama
from langchain.embeddings import OllamaEmbeddings
from langchain.chains import RetrievalQA
from langchain.text_splitter import CharacterTextSplitter
from langchain.docstore.document import Document


load_dotenv()

True

In [2]:
# Load & chunk the Sikka API docs
df = pd.read_csv("Sikka_APIs - Sikka_APIs.csv", encoding="utf-8")

In [3]:
df.describe()

,API Name,Description,API Endpoints,Document Link
count,93,93,93,93
unique,93,89,93,92
top,appointments,Returns appointments data from practice,https://api.sikkasoft.com/v4/appointments,https://apidocs.sikkasoft.com/#9086881d-f5cc-4...
freq,1,2,1,2


In [4]:
df.head(5)

,API Name,Description,API Endpoints,Document Link
0,appointments,Returns appointments data from practice,https://api.sikkasoft.com/v4/appointments,https://apidocs.sikkasoft.com/#cc4375ec-0b6a-4...
1,appointments_available_slots,Returns available appointments slots from prac...,https://api.sikkasoft.com/v4/appointments_avai...,https://apidocs.sikkasoft.com/#6b18ec4a-cfec-4...
2,accounts_receivables,Returns account receivables details from practice,https://api.sikkasoft.com/v4/accounts_receivables,https://apidocs.sikkasoft.com/#fcb005b7-b9ce-4...
3,accounts_receivables_by_patients,Returns account receivables details by patient...,https://api.sikkasoft.com/v4/accounts_receivab...,https://apidocs.sikkasoft.com/#3ae94fe1-4b24-4...
4,patients,Returns list of patients in practice,https://api.sikkasoft.com/v4/patients,https://apidocs.sikkasoft.com/#67aa35de-a4d4-4...


In [5]:
# Combine Columns into Formatted Strings
rows = []
for _, row in df.iterrows():
    combined = " | ".join([
        f"API Name: {row['API Name']}",
        f"Description: {row['Description']}",
        f"Endpoint: {row['API Endpoints']}",
        f"Docs: {row['Document Link']}"
    ])
    rows.append(combined)

# Create Raw Text Document
raw_text = "\n".join(rows)

# Split into Chunks for LLM Input
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = text_splitter.split_documents([Document(page_content=raw_text)])

In [6]:
# Creating vector store with FAISS

embeddings = OllamaEmbeddings(model="mistral")
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()

C:\Users\chira\AppData\Local\Temp\ipykernel_19084\3665739574.py:3: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="mistral")


In [7]:
# Initialize language model

llm = Ollama(model="mistral") 

C:\Users\chira\AppData\Local\Temp\ipykernel_19084\2761387838.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(model="mistral")


In [8]:
# RAG QA chain
retrieval_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff"
)

In [9]:
# Question 1
query = "List top 5 API names?"
response = retrieval_chain.run(query)

# Print the Result
print("\nQuery:", query)
print("\nResponse:", response)

C:\Users\chira\AppData\Local\Temp\ipykernel_19084\913169501.py:3: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = retrieval_chain.run(query)



Query: List top 5 API names?

Response: 1. finance/balances
   2. finance/payroll_details
   3. finance/employee
   4. finance/vendors
   5. finance/budgets


In [10]:
# Question 2
query = "Which API endpoint can be used to retrieve patient payment details?"
response = retrieval_chain.run(query)

# Output Result
print("\nQuery:", query)
print("\nResponse:", response)



Query: Which API endpoint can be used to retrieve patient payment details?

Response:  The endpoint for retrieving patient payment details from the given list of APIs is not explicitly mentioned. However, based on the context of finance APIs provided, you might want to check the `finance/transactions` endpoint as it returns transaction details from the practice's finance system, which could potentially include patient payment information.

Here is the relevant information for reference:

API Name: finance/transactions
Description: Returns transaction details from practice's finance system.
Endpoint: https://api.sikkasoft.com/v4/finance/transactions
Docs: https://apidocs.sikkasoft.com/#3b10df23-07b9-4335-bbfb-78eae1393654


In [11]:
# Question 3
query = "Get patient payments"
response = retrieval_chain.run(query)

# Output Result
print("\nQuery:", query)
print("\nResponse:", response)


Query: Get patient payments

Response:  Based on the provided APIs, it seems that there isn't a specific endpoint for "patient payments" in the given list. However, you can get related information from the following endpoints:

1. `finance/transactions`: This API returns transaction details from the practice's finance system. You might find payment-related transactions here.

2. `finance/balances`: This API returns balance details data from the practice's finance system, which could include patient balances or payments.

To narrow down your search for specific payment-related information, you should consider the type of data you are looking for (e.g., payment date, payment amount, patient ID, etc.). By understanding what data is available in each endpoint and adjusting your request accordingly, you can gather the necessary information to answer your question.


In [12]:
# Question 4
query = "AI system to develop a Payment System, referencing existing solutions such as Nadapayments for requirements."
response = retrieval_chain.run(query)

# Output Result
print("\nQuery:", query)
print("\nResponse:", response)



Query: AI system to develop a Payment System, referencing existing solutions such as Nadapayments for requirements.

Response:  To develop a payment system for the practice, you can leverage existing solutions like Nadapayments and customize them according to your specific needs. Here's a list of APIs from the provided Sikkasoft platform that would be helpful in creating the desired payment solution:

1. finance/customers - Get customer details data from the practice's finance system, which will help identify who needs to make payments.
2. finance/transactions - Retrieve transaction details from the practice's finance system, useful for tracking payments and transactions related to the practice.
3. finance/balances - Use this API to obtain balance details data from the practice's finance system, so that users can see their account balances before making a payment.
4. finance/payroll_details - Although it might not be directly related to the payment system, understanding employee-relat

In [3]:
df = pd.read_html('https://en.wikipedia.org/wiki/List_of_The_Simpsons_characters')

In [7]:
len(df)

6